# Gray baselines on identical draws (CPU): SCUNet and DRUNet on Set12 / BSD68, seeds 0–2

These are the released supervised gray denoisers, run on exactly the noisy images of `gray-lift-kill-test`, for
paired tests against the chroma-noise lift.
* Protocol: Restormer / KAIR gray. The float32 image is divided by 255, `np.random.seed(seed)` is set per image, then
  `img += np.random.normal(0, σ/255)`. No clipping; outputs are clamped and rounded to uint8.
* SCUNet, σ-specific gray weights from KAIR v1.0, whole image, no x8 self-ensemble. Its published BSD68 numbers are
  31.99 / 29.55 / 26.67 dB, the last being the best σ 50 result without an external prior.
* DRUNet gray: KAIR v1.0 weights with a noise-level map.
The seed-0 rerun must land within ±0.03 dB of the published numbers. Kaggle: CPU, internet on.

In [ ]:
import importlib.util, math, os, sys, time, types, urllib.request
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

SIGMAS, SEEDS = [15, 25, 50], [0, 1, 2]
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
torch.set_num_threads(os.cpu_count())
T_START = time.time()


def download(url, path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        for attempt in range(5):
            try:
                urllib.request.urlretrieve(url, path)
                break
            except Exception as e:
                print(f"  retry {attempt + 1} for {url}: {e}", flush=True)
                time.sleep(5 * (attempt + 1))
    return path


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


KAIR = "https://raw.githubusercontent.com/cszn/KAIR/fc1732f4a4514e42ce15e5b3a1e18c828af47a1e/{}"
SCUNET = "https://raw.githubusercontent.com/cszn/SCUNet/52e440a80a655b01e0b41e9dd9bfe599bc11625e/{}"
REL = "https://github.com/cszn/KAIR/releases/download/v1.0/{}"
RAW = "https://raw.githubusercontent.com/cszn/DnCNN/master/testsets/{}/{}"
IMAGES = {"Set12": [f"{i:02d}.png" for i in range(1, 13)], "BSD68": [f"test{i:03d}.png" for i in range(1, 69)]}
os.makedirs(f"{CACHE}/models", exist_ok=True)
jobs = {f"{CACHE}/models/basicblock.py": KAIR.format("models/basicblock.py"),
        f"{CACHE}/models/network_unet.py": KAIR.format("models/network_unet.py"),
        f"{CACHE}/network_scunet.py": SCUNET.format("models/network_scunet.py"),
        f"{CACHE}/drunet_gray.pth": REL.format("drunet_gray.pth")}
jobs.update({f"{CACHE}/scunet_gray_{s}.pth": REL.format(f"scunet_gray_{s}.pth") for s in SIGMAS})
jobs.update({f"{CACHE}/{ds}_{n}": RAW.format(ds, n) for ds, ns in IMAGES.items() for n in ns})
with ThreadPoolExecutor(8) as ex:
    list(ex.map(lambda kv: download(kv[1], kv[0]), jobs.items()))
open(f"{CACHE}/models/__init__.py", "w").close()
sys.path.insert(0, CACHE)
from models.network_unet import UNetRes
if importlib.util.find_spec("thop") is None:  # SCUNet imports thop only for a FLOP count
    sys.modules["thop"] = types.ModuleType("thop")
    sys.modules["thop"].profile = None
SCUNet = load_module("network_scunet", f"{CACHE}/network_scunet.py").SCUNet
TEST = {ds: [(n, np.asarray(Image.open(f"{CACHE}/{ds}_{n}").convert("L"))) for n in ns] for ds, ns in IMAGES.items()}
NETS = {}
for s in SIGMAS:
    NETS[f"SCUNet {s}"] = SCUNet(in_nc=1, config=[4] * 7, dim=64)
    NETS[f"SCUNet {s}"].load_state_dict(torch.load(f"{CACHE}/scunet_gray_{s}.pth", map_location="cpu", weights_only=True), strict=True)
NETS["DRUNet"] = UNetRes(in_nc=2, out_nc=1, nc=[64, 128, 256, 512], nb=4, act_mode="R", downsample_mode="strideconv",
                         upsample_mode="convtranspose", bias=False)
NETS["DRUNet"].load_state_dict(torch.load(f"{CACHE}/drunet_gray.pth", map_location="cpu", weights_only=True), strict=True)
for net in NETS.values():
    net.eval().requires_grad_(False)
print({ds: len(v) for ds, v in TEST.items()}, list(NETS), f"setup {(time.time() - T_START) / 60:.1f} min")

In [ ]:
def noisy_of(clean_u8, sigma, seed):
    img = np.float32(clean_u8[..., None]) / 255.
    np.random.seed(seed=seed)
    img += np.random.normal(0, sigma / 255., img.shape)
    return torch.from_numpy(img[..., 0].copy())


def psnr(x, clean_u8):
    a = (x.clamp(0, 1) * 255).round().double()
    return 10 * math.log10(255 ** 2 / (a - torch.from_numpy(clean_u8).double()).pow(2).mean().item())


@torch.no_grad()
def scunet(net, y):
    return net(y[None, None])[0, 0]


@torch.no_grad()
def drunet(net, y, sigma01):
    h, w = y.shape
    yp = F.pad(y[None, None], (0, (-w) % 8, 0, (-h) % 8), mode="replicate")
    return net(torch.cat([yp, torch.full_like(yp, sigma01)], 1))[0, 0, :h, :w]


rows = []
for seed in SEEDS:
    for s in SIGMAS:
        for ds, items in TEST.items():
            for name, clean in items:
                y = noisy_of(clean, s, seed)
                rows.append(dict(dataset=ds, image=name, sigma=s, seed=seed, method="SCUNet-gray", psnr=psnr(scunet(NETS[f"SCUNet {s}"], y), clean)))
                rows.append(dict(dataset=ds, image=name, sigma=s, seed=seed, method="DRUNet-gray", psnr=psnr(drunet(NETS["DRUNet"], y, s / 255), clean)))
        print(f"seed {seed} σ {s} done, {(time.time() - T_START) / 60:.1f} min", flush=True)
SCORES = pd.DataFrame(rows)
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
print(SCORES.groupby(["dataset", "method", "seed", "sigma"]).psnr.mean().unstack().round(3).to_string())
print("published: SCUNet BSD68 31.99 / 29.55 / 26.67, Set12 33.43 / 31.09 / 28.04; DRUNet BSD68 31.91 / 29.48 / 26.59, Set12 33.25 / 30.94 / 27.90")
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))